# Initial Observations

- The dataset has already been cleaned and validated.
- The objective of this notebook is to create new business features that improve analysis and dashboard reporting.
- Feature engineering helps transform raw data into meaningful business metrics.
- The newly created features will be used in Python visualizations and Power BI dashboards.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../Dataset/processed/zomato_final.csv")

df.head()

,name,location,cuisines,rest_type,online_order,book_table,rate,votes,approx_cost,listed_in_type,listed_in_city
0,Jalsa,Banashankari,"North Indian, Mughlai, Chinese",Casual Dining,Yes,Yes,4.10,775,800.0,Buffet,Banashankari\n
1,Spice Elephant,Banashankari,"Chinese, North Indian, Thai",Casual Dining,Yes,No,4.10,787,800.0,Buffet,Banashankari\n
2,San Churro Cafe,Banashankari,"Cafe, Mexican, Italian","Cafe, Casual Dining",Yes,No,3.80,918,800.0,Buffet,Banashankari\n
3,Addhuri Udupi Bhojana,Banashankari,"South Indian, North Indian",Quick Bites,No,No,3.70,88,300.0,Buffet,Banashankari\n
4,Grand Village,Basavanagudi,"North Indian, Rajasthani",Casual Dining,No,No,3.80,166,600.0,Buffet,Banashankari\n


In [3]:
df["Price_Category"] = pd.cut(
    df["approx_cost"],
    bins=[0,300,700,10000],
    labels=["Budget","Mid-Range","Premium"]
)

In [4]:
df["Price_Category"].value_counts()

Price_Category
Mid-Range    21894
Budget       18515
Premium      10856
Name: count, dtype: int64

In [14]:
df["Rating_Category"] = pd.cut(
    df["rate"],
    bins=[0,3,3.5,4,5],
    labels=[
        "Poor",
        "Average",
        "Good",
        "Excellent"
    ]
)

In [16]:
df["Rating_Category"].value_counts()

Rating_Category
Good         18142
Average      10986
Excellent     9184
Poor          3278
Name: count, dtype: int64

In [17]:
df["Popularity_Category"] = pd.qcut(
    df["votes"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [18]:
df["Popularity_Category"].value_counts()

Popularity_Category
Low          13669
Very High    12875
High         12807
Medium       12258
Name: count, dtype: int64

In [19]:
df["Service_Type"] = np.select(
    [
        (df["online_order"] == "Yes") & (df["book_table"] == "Yes"),
        (df["online_order"] == "Yes") & (df["book_table"] == "No"),
        (df["online_order"] == "No") & (df["book_table"] == "Yes")
    ],
    [
        "Full Service",
        "Online Focused",
        "Dine-In Focused"
    ],
    default="Basic Service"
)

In [20]:
df["Cuisine_Count"] = df["cuisines"].fillna("").apply(
    lambda x: len([c.strip() for c in x.split(",") if c.strip()])
)

In [21]:
df["Popularity_Score"] = (
    df["rate"] * np.log1p(df["votes"])
).round(2)

In [22]:
df["Cuisine_Count"] = df["cuisines"].fillna("").apply(
    lambda x: len([c.strip() for c in x.split(",") if c.strip()])
)

df["Cuisine_Count"].head()

0    3
1    3
2    3
3    2
4    2
Name: Cuisine_Count, dtype: int64

In [23]:
df["Cost_per_Vote"] = (
    df["approx_cost"] / df["votes"]
).replace([np.inf, -np.inf], np.nan)

df["Cost_per_Vote"] = df["Cost_per_Vote"].round(2)

In [24]:
df["Multi_Cuisine"] = np.where(
    df["Cuisine_Count"] > 1,
    "Yes",
    "No"
)

df["Multi_Cuisine"].value_counts()

Multi_Cuisine
Yes    39187
No     12422
Name: count, dtype: int64

In [25]:
df["Premium_Restaurant"] = np.where(
    df["approx_cost"] >= 1000,
    "Yes",
    "No"
)

df["Premium_Restaurant"].value_counts()

Premium_Restaurant
No     44720
Yes     6889
Name: count, dtype: int64

In [26]:
df["Highly_Rated"] = np.where(
    df["rate"] >= 4.0,
    "Yes",
    "No"
)

df["Highly_Rated"].value_counts()

Highly_Rated
No     39245
Yes    12364
Name: count, dtype: int64

In [27]:
df["Popularity_Score"] = (
    df["rate"] * np.log1p(df["votes"])
).round(2)

df["Popularity_Score"].head()

0    27.28
1    27.34
2    25.93
3    16.61
4    19.45
Name: Popularity_Score, dtype: float64

In [28]:
df.to_csv(
    "../Dataset/processed/zomato_featured.csv",
    index=False
)

# Key Findings

- Created price segments to classify restaurants into Budget, Mid-Range, and Luxury categories.
- Categorized restaurants based on customer ratings and popularity.
- Identified restaurant service models using online ordering and table booking.
- Calculated the number of cuisines offered by each restaurant.
- Created a Cost per Vote metric to evaluate pricing efficiency.
- Identified premium and highly rated restaurants using business rules.
- Developed a Popularity Score by combining customer ratings and votes.
- Exported the feature-engineered dataset for exploratory analysis, visualization, and Power BI dashboard development.